In [ ]:
print("working!")

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
!pip install -r requirements.txt

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects , return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# chunking operation
# split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)

In [ ]:
len(texts_chunk)

In [ ]:
texts_chunk

In [ ]:
# Embedding model
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )

    return embeddings

embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
# testing embedding model
vector = embedding.embed_query("How are you?")
print(vector)

In [ ]:
len(vector)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# XAI_API_KEY = os.getenv("XAI_API_KEY")
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
# os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# os.environ["XAI_API_KEY"] = XAI_API_KEY
# os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name= index_name,
        dimension= 384, # dimension of the embedding
        metric= "cosine", # cosine similarity metric
        spec= ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
# store vectors in vector db
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
# Add more data to existing Pinecone Index
# to_add = Document(
#     page_content="Sample content to be added",
#     metadata={"source": "personal"}
# )

In [ ]:
# connect the retriever
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
from langchain.globals import set_debug
set_debug(True)

In [ ]:
retrieved_docs = retriever.invoke("What is acne?")
# retrieved_docs

In [ ]:
# # from langchain_openai import ChatOpenAI
# from langchain.chat_models import init_chat_model

# # chatModel = ChatOpenAI(model="gpt-4o")
# chatModel = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [ ]:
from langchain.globals import set_verbose
set_verbose(True)

In [ ]:
# from langchain_openai import ChatOpenAI

# chatModel = ChatOpenAI(
#     model="google/gemini-2.5-flash:free",
#     openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
#     base_url="https://openrouter.ai/api/v1",
#     temperature=0
# )

In [ ]:
# !pip install -U langchain-xai

In [ ]:
# !pip install -U langchain-groq

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_xai import ChatXAI

# os.environ["XAI_API_KEY"] = os.getenv("XAI_API_KEY")

# chatModel = ChatOpenAI(
#     model="grok-4.3",
#     openai_api_key=os.environ["XAI_API_KEY"],
#     openai_api_base="https://api.x.ai/v1",
#     temperature=0
# )

In [ ]:
# import os
# from langchain_groq import ChatGroq

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# chatModel = ChatGroq(
#     model="llama-3.1-8b-instant",
#     temperature=0
# )

In [ ]:
import os
import langchain
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Patch the internal cache bug
langchain.llm_cache = None

load_dotenv()

# Using Meta's free-tier Llama model via OpenRouter
chatModel = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct:free",
    openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are a medical assistant for answering to tasks. "
    "Use the following pieces of retrieved context to answer questions. "
    "If you don't know the answer, simply respond that you don't know. "
    "Use five sentences at maximum and keep the response concise. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
import os
import langchain
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

langchain.llm_cache = None

In [ ]:
response = rag_chain.invoke({"input": "What is Acromegaly and Gigantism?"})
print(response["answer"])

In [ ]:
import os
import langchain
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Patch the internal cache bug
langchain.llm_cache = None

# 2. Load environment variables
load_dotenv()

# 3. Use the universal free model router (automatically finds an active free model)
chatModel = ChatOpenAI(
    model="openrouter/free",
    openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

# 4. Define prompt
system_prompt = (
    "You are a medical assistant for answering to tasks. "
    "Use the following pieces of retrieved context to answer questions. "
    "If you don't know the answer, simply respond that you don't know. "
    "Use five sentences at maximum and keep the response concise. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 5. Rebuild the chains with the updated chatModel
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 6. Execute the query
response = rag_chain.invoke({"input": "What is Acromegaly and Gigantism?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is acne?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is the treatment of acne?"})
print(response["answer"])